### Writing to Sentinel1-sig0 zarr store

Necessary imports

In [ ]:
import pystac_client as pc
import xarray as xr
import zarr
import numpy as np
import rioxarray
import pandas as pd
from datetime import datetime
from collections import defaultdict

In [ ]:
import sys

if not sys.warnoptions:
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning)

#### Useful functions

We need some functions to make the writing process easier

In [ ]:
def group_by_relative_orbit(items, key="sat:relative_orbit"):
    """
        Function to group a list of stac items by their relative orbit. Result is a dictionaray with rel orbit as keys and items as values.
    """
    groups = defaultdict(list)
    for it in items:
        groups[it[0].properties[key]].append(it)
    return dict(groups)

In [ ]:
def get_idx(array1, array2):
    """
        Function to get the indices of coordinates from an array in a different array. To be used when wanting to write to a zarr store with indexing.
    """
    min = np.where(array1==array2[0])[0][0]
    max = np.where(array1==array2[-1])[0][0]+1
    return min, max

In [ ]:
def load_data(item, pols):
    """
        Load an item and add the sensing date as a time dimension, also rename acoording the polarization. If multiple polarizations are to be used a merges dataset will be created.
    """
    if type(pols)==str:
        data = rioxarray.open_rasterio(item.assets[pols].href).compute().expand_dims(time=pd.to_datetime([item.properties["datetime"]]).tz_convert(None)).rename(pols)
    else:
        data = []
        for pol in pols:
            data.append(rioxarray.open_rasterio(item.assets[pol].href).compute().expand_dims(time=pd.to_datetime([item.properties["datetime"]]).tz_convert(None)).rename(pol))
        
        data = xr.merge(data)
    return data.squeeze()

In [ ]:
def get_datetime(item):
    """
        Get the datetime from an item in the correct format
    """
    return datetime.strptime(item.properties["datetime"], "%Y-%m-%dT%H:%M:%SZ")

In [ ]:
def group_dates(item_list):
    """
        Group stac items dependant on their sensing dates. If sensing dates are close enough so the items are from the same orbit the will be grouped in a list of lists.
    """
    
    # Initialize list in list
    grouped_items = [[]]
    i=0


    for item in item_list:
        
        # If list in grouped items is empty append the item
        if not grouped_items[i]:
            grouped_items[i].append(item)
        
        # Otherwise check if the item is close enough to the existing item in the list and append if thats the case, if not append a new list to the grouped items with that item
        else: 
            if get_datetime(item) - get_datetime(grouped_items[i][-1]) <= pd.Timedelta(seconds=100):
                grouped_items[i].append(item)

            else:
                grouped_items.append([item])
                i+=1

    return grouped_items

In [ ]:
def read_and_merge_items(items, pols):
    """
        Use the function load_data on grouped items and combine them according to the group. Reads items and combines data with a close enough sensing date to a single array, then combines the arrays to a large dataset. Works for a single polarization or a list of polarizations to be read.
    """
    first = True

    if type(pols)==list:
        datasets = []
        for pol in pols:
            # Iterate through a group of items and load the data according to the polarization
            for item in items:
                ds = load_data(item, pol)
                
                # If first data which is read, rename 
                if first:
                    data = ds
                    first = False
                
                # Otherwise the nodata values of the existing array are overwritten with the new data
                else:
                    data = xr.where(data==-9999, ds, data, keep_attrs=True)

            # append the combined array to a list of arrays, expand the time dim if not there already
            if "time" in data.dims:      
                datasets.append(data)
            else:
                datasets.append(data.expand_dims(time=pd.to_datetime([item.properties["datetime"]]).tz_convert(None)))

            # Reset and restart for the next group
            first=True

        # Merge the datasets to a large dataset
        data = xr.merge(datasets)

    # Works similar but only for one polarization
    else:
        for item in items:
            ds = load_data(item, pols)
            
            if first:
                data = ds
                first = False
            
            else:
                data = xr.where(data==-9999, ds, data, keep_attrs=True)

        data = data.to_dataset(name=pols)

    # Return the squeezed dataset
    return data.squeeze()

#### Workflow

**Get items from EODC stac catalogue**

First we need the get items from a the EODC catalogue. The zarr store is only available over Austria and a chunk is for one month. So we will get items for a tile over Austria and for one month.  

In [ ]:
pc_client = pc.Client.open("https://stac.eodc.eu/api/v1")

time_range = "2024-01-01/2024-02-01"

search = pc_client.search(
    collections=["SENTINEL1_SIG0_20M"],
    datetime= time_range,
    query={"Equi7_TileID": {"eq": f"EU020M_E048N015T3"}})

items_eodc = search.item_collection()

Next we will sort them so that they are sorted by ascending time. Also the items are grouped so that items from the same satellite scene (similar sensing time) are grouped together in a list.

In [ ]:
item_list = list(items_eodc)[::-1]
grouped_items = group_dates(item_list)

As the zarr store has a dimension for the relative orbit numbers, the items are also grouped by their rel orbit numbers. The result is a dict with the the relative orbit numbers as keys and the respective items as values (Note that the items are already grouped by their sensing date, so the values are groups of items)

In [ ]:
grouped_orbits = group_by_relative_orbit(grouped_items)

**Open zarr store**

Next we can open the zarr store and load some necessery data from it

In [ ]:
store = zarr.storage.LocalStore("s1sig0.zarr")
group = zarr.group(store=store)

x_extent = group["x"][:]
y_extent = group["y"][:]
rel_orbit_extent = group["relative_orbit_number"][:]

As we need to write the sensing date as its own dataarray in the zarr store, we have to define an origin, to be able to write the data in int format (seconds since origin)

In [ ]:
sensing_origin = np.datetime64("2014-10-01T00:00:00")

Also we can get the start and end date of the data to be written to the store

In [ ]:
start = np.datetime64(time_range.split("/", 1)[0].strip(), "D")
end = np.datetime64(time_range.split("/", 1)[1].strip(), "D")

Next we will only process one relative orbit. We need to get the index of the relative orbit in the zarr store.

In [ ]:
orbit = 22
orbit_index = np.where(rel_orbit_extent==orbit)[0][0]

Now we can iterate through the items of that orbit, read the data and merge the items in the same sensing time group

In [ ]:
datasets_orbits=[]

for items in grouped_orbits[orbit]:
    ds = read_and_merge_items(items, ["VV", "VH"])

    # Get the relative orbit number to be a dimesion
    ds = ds.expand_dims({"rel_orbit_number": [ds.attrs["rel_orbit_number"]]})

    # The sensing date and absolute orbit number will be data variables
    ds["sensing_date"] = (ds['time'].values.astype("datetime64[s]") - sensing_origin).astype("int64")
    ds["abs_orbit_number"] = ds.attrs["abs_orbit_number"]

    # The time dimension will be in only be stored without the time. Only Day of sensing is used 
    ds['time'] = ds['time'].astype('datetime64[D]')

    # Append the read dataset to a list of datasets
    datasets_orbits.append(ds)

Now the datasets in the datasets_orbits list can be concatenated along the time dimension

In [ ]:
combined_orbits = xr.concat(datasets_orbits, dim="time", combine_attrs="override")

As some time coordinates will be missing in the resulting dataset, we will add them and fill the data with nodata values

In [ ]:
full_times = pd.date_range(start=start, end=end, freq='D')
result = combined_orbits.reindex(time=full_times, fill_value=-9999)

For the next steps we can transpose the dataset

In [ ]:
result = result.transpose("rel_orbit_number", "time", "y", "x")

To write the sensing date and abolutes orbit numbers as their own arrays we need to make sure they are 4-dimensional

In [ ]:
sensing_dates = result["sensing_date"].values.reshape(1,result.sizes["time"],1,1)
abs_orbit_numbers = result["abs_orbit_number"].values.reshape(1,result.sizes["time"],1,1)

Also when reading the items, coordinates are changed to center of pixel, this is changed to be edge of pixel

In [ ]:
result["x"] = result.x-10
result["y"] = result.y+10

Now we can get the indexes of the x and y coordinates in the zarr store corresponding to the coordinates of our data.

In [ ]:
x_min, x_max = get_idx(x_extent, result["x"].values)
y_min, y_max = get_idx(y_extent, result["y"].values)

And the same for our time indexes

In [ ]:
time_origin = np.datetime64("2014-10-01")
time_min = (result.time.min().values.astype("datetime64[D]") - time_origin).astype("int64")
time_max = (result.time.max().values.astype("datetime64[D]") - time_origin).astype("int64")+1

Finally, we can write the data to the correct spot in the zarr store

In [ ]:
group["VH"][orbit_index:orbit_index+1,time_min:time_max, y_min:y_max, x_min:x_max] = result["VH"].values
group["VV"][orbit_index:orbit_index+1,time_min:time_max, y_min:y_max, x_min:x_max] = result["VV"].values

To write the sensing date and absolute orbit number we still need them to have the correct shape

In [ ]:
sensing_dates = np.broadcast_to(sensing_dates, (1,time_max-time_min, y_max-y_min, x_max-x_min))
abs_orbit_numbers = np.broadcast_to(abs_orbit_numbers, (1,time_max-time_min, y_max-y_min, x_max-x_min))

Then we can write them as well

In [ ]:
group["sensing_date"][orbit_index:orbit_index+1,time_min:time_max, y_min:y_max, x_min:x_max] = sensing_dates
group["absolute_orbit_number"][orbit_index:orbit_index+1,time_min:time_max, y_min:y_max, x_min:x_max] = abs_orbit_numbers